<a href="https://colab.research.google.com/github/rsghotra/1-ultra-low-latency-java/blob/main/Chapter%201/chapter_1_lab_3_non-agentic_vs_agentic-workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 - Lab 3: <font color='blue'>Agentic vs non-Agentic workflow</font>

**<font color='purple'>Goal</font>**:
In this lab, you'll compare **Agentic** and **non-agentiw workflow**:

* Chatting with an LLM without access to external resources
* Chatting with an LLM with augmented information using external APIs:
    -  Fetch Historical Prices for a given stock (The same use case in the book)
    -  Collect News (Additional use case to practice)

**<font color='purple'>Tech stack</font>**:

We'll use :
* OpenAI Responses API ==> Standard calls + Function Callings
* OpenAI Agent SDK ==> Simple Agent Abstraction
* NewsApi: https://newsapi.org/

You need to create API Keys in both OpenAI (you need to add some credits) and NewsAPI (free until a certain # of calls).

Knowledge Cutoff of GPT-4.1 is June 2024

## Install packages

Install openai and newsapi packages

In [1]:
%pip install openai
%pip install newsapi-python

Add you OpenAI Key in Google colab: in the left vertical menu, you have an icon key. Click on it, and add your OpenAI key with a name

In [2]:
import os

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    NEWS_API_KEY = userdata.get('NEWS_API_KEY')
except ImportError:
    # Running locally - read the keys from the environment (see ../.env.sample).
    OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
    NEWS_API_KEY = os.environ['NEWS_API_KEY']

To have more information on how to call the API, you can read the documentation below.

In the various labs, you'll have a detailed overview on how to use the API.

https://platform.openai.com/docs/api-reference



# 1- Non Agentic Workflow

## Historical Price

### Responses API - GPT-4.1

https://platform.openai.com/docs/api-reference/responses

In [3]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
    model="gpt-4.1",
    input="What is the last price of NVIDIA."
)

print(response)

Response(id='resp_03a9e8500fda8f73006b07acb637ac87d2b5d45edb2544b48b', created_at=1795665078.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_03a9e8500fda8f73006b07acb7102887d282506ac29e4df6cf', content=[ResponseOutputText(annotations=[], text="As of my most recent update in June 2024, I do not have access to live financial data. **NVIDIA's (NVDA) stock price frequently changes during market hours.** \n\nTo get the latest price:\n\n- Check financial news websites (Yahoo Finance, Google Finance, Bloomberg).\n- Use a stock brokerage app.\n- Search “NVIDIA stock price” in your browser.\n\nIf you need a price from a particular date or time, let me know!", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1795665079.0, 

In [5]:
print(response.output[0].content[0].text)

As of my most recent update in June 2024, I do not have access to live financial data. **NVIDIA's (NVDA) stock price frequently changes during market hours.** 

To get the latest price:

- Check financial news websites (Yahoo Finance, Google Finance, Bloomberg).
- Use a stock brokerage app.
- Search “NVIDIA stock price” in your browser.

If you need a price from a particular date or time, let me know!


In [4]:
print(response.output[0].content[0].text)

As of my most recent update in June 2024, I do not have access to live financial data. **NVIDIA's (NVDA) stock price frequently changes during market hours.** 

To get the latest price:

- Check financial news websites (Yahoo Finance, Google Finance, Bloomberg).
- Use a stock brokerage app.
- Search “NVIDIA stock price” in your browser.

If you need a price from a particular date or time, let me know!


### Results

As yu can see, the LLM was not able to provide the last price of NVIDIA, as its knwoledge cufoff was on June 2024.

### Another Try

In this part, I'll provide a web site link. let's see what happens:

In [6]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response2 = client.responses.create(
  model="gpt-4.1",
  input="What is the last price of NVIDIA. Can you access to https://finance.yahoo.com/quote/NVDA to collect the last price? "
)

print(response2.output[0].content[0].text)

I can't browse the web in real time or access external websites directly, including finance.yahoo.com. However, you can view the latest price of NVIDIA (NVDA) stock by visiting the [Yahoo Finance NVDA page](https://finance.yahoo.com/quote/NVDA) directly.

If you want, I can show you how to retrieve the latest price using Python with financial APIs, or suggest trustworthy sources to check live stock prices. Just let me know how you'd like to proceed!


I have the same issue, the LLM cannot provide the price because it cannot access and browse external websites like this.

## News

### Responses API

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1",
  input="Give me the latest news about NVIDIA."
)

print(response.output[0].content[0].text)

Here are some of the latest news highlights about **NVIDIA** as of June 2024:

---

### 1. **NVIDIA Surpasses $3 Trillion Market Cap**
- NVIDIA recently became the world’s second most valuable company, surpassing Apple with a market capitalization above $3 trillion. This surge is driven by ongoing demand for NVIDIA's AI and data center chips.

### 2. **NVIDIA Unveils Next-Gen Rubin GPUs**
- At Computex 2024, NVIDIA revealed its new **Rubin GPU architecture**, which is expected to power its next line of AI accelerators and graphics cards. This follows their highly successful Blackwell GPUs.

### 3. **AI and Data Center Growth**
- NVIDIA's Q1 2024 earnings report highlights explosive growth in its data center segment, mainly due to soaring demand for AI hardware among companies building large language models and generative AI services.

### 4. **Partnerships with Major Tech Firms**
- NVIDIA is expanding partnerships with companies like Microsoft, Amazon, and Google to provide AI infrastr

### Results

If you search for "NVIDIA Surpasses $3 Trillion Market Cap" in the web, you'll see that this news is from June 2024... as it's mentionned by the llm itself in the output.

# 2- Agentic Workflow:

In the following implementation, we'll use tools call to avoid knowledge cutoff and therefore build a basic agentic systeme

## Tools Calling without agent abstraction: Historical

Here, you'll learn how to use function calling with Responses API, without using any agent abstraction

1. You need to define what we call "Tools" object: Type (function), name, Description, parameters...
2. Declare your function: Tool to access external API
3. Call Responses API by enabling: "tools"

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

In [13]:
import yfinance as yf
from openai import OpenAI
import json
import os

client = OpenAI(api_key = OPENAI_API_KEY)



#1. Define a list of callable tools for the model
tools = [
    {
        "type":"function",
        "name":"get_latest_stock_price",
        "description":"Fetch the current stock price for the give symbol.",
        "parameters": {
            "type":"object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "A ticker symbol of a givens stock",
                },
            },
            "required": ["symbol"],
        },
    },
]

#2 Define the tool
def get_latest_stock_price(symbol: str) -> dict:
  """
    Fetch the current stock price for the given symbol.
  """

  ticker = yf.Ticker(symbol)
  hist = ticker.history(period="1d")
  latest_close = hist['Close'].iloc[-1]
  latest_date = hist.index[-1].strftime('%Y-%m-%d')

  return {'price': latest_close, 'date': latest_date}

input_list = [
    {"role": "user", "content": "What is the current price of nvidia."}
]

#2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-4.1-mini",
    tools = tools,
    input = input_list,
)

# 3. Save function call outputs of subsequent requests
input_list += response.output

for item in response.output:
  if item.type == "function_call":
    print(json.loads(item.arguments))
    # 4. Execute the function logic
    prices = get_latest_stock_price(json.loads(item.arguments)['symbol'])
    print(prices)

    #5. Provide function call results to the model
    input_list.append({
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": json.dumps({
            "prices": prices
        })
    })

print("Final input: ")
print(input_list)

# 6. Call the API with the updated input list so it can prepare a response
response = client.responses.create(
    model = "gpt-4.1-mini",
    instructions = "Respond only with price and date generated by a tool.",
    tools = tools,
    input = input_list,
)

#7. The model should be able to give a response!
print("Final output: ")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)


{'symbol': 'NVDA'}
{'price': np.float64(149.3000030517578), 'date': '2026-11-25'}
Final input: 
[{'role': 'user', 'content': 'What is the current price of nvidia.'}, ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_9FUBmQTQCJXM6Ood3dxok8hB', name='get_latest_stock_price', type='function_call', id='fc_072b4eadd7bb78ad006b07c565386087d2965911cf4f7573ad', caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_9FUBmQTQCJXM6Ood3dxok8hB', 'output': '{"prices": {"price": 149.3000030517578, "date": "2026-11-25"}}'}]
Final output: 
{
  "id": "resp_072b4eadd7bb78ad006b07c5659da887d293ed25767d647ee9",
  "created_at": 1795671397.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with price and date generated by a tool.",
  "metadata": {},
  "model": "gpt-4.1-mini-2025-04-14",
  "object": "response",
  "output": [
    {
      "id": "msg_072b4eadd7bb78ad006b07c56604a087d28f1ca54acadaf742",
      "content

* This will generate a ResponseFunctionToolCall with the name of the function, and the argument (well interpreted by the model, it gives a symbol instead of the name of the company)
* We then to execute this function with the right argument


* Send back the whole result to the LLM to interpret and output the final answer

https://composio.dev/blog/claude-function-calling-tools

## OpenAI Agent SDK

In this part, we'll be using OpenAI Agent SDK.

You need first to install the package.

We'll create:
*  A tool: to fetch historical prices
*  An agent:
   - Describe the instructions to give to the agent
   - List of the tools the agent can use
   - Model

Install OpenAI Agents SDK

In [14]:
!pip install openai-agents


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.6 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [openai-agents]


In [27]:
import os
import yfinance as yf
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

#we import the Agent components
from agents import Agent, Runner, function_tool
import nest_asyncio
nest_asyncio.apply()

#create a tool to fecth historical prices
@function_tool
def get_latest_stock_price(symbol: str) -> dict:
  """
  Fetch the current stock price for the given symbol.
  """
  ticker = yf.Ticker(symbol)
  hist = ticker.history(period="1d")
  latest_close = hist['Close'].iloc[-1]
  latest_date = hist.index[-1]

  return {'price': latest_close, 'date': latest_date}

ticker = yf.Ticker("NVDA")
hist = ticker.history(period="1d")
hist


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-11-25 00:00:00-05:00,144.059998,150.369995,141.169998,149.300003,363377300,0.0,0.0


### Historical Prices Agent

#### Tool

➡ Create a tool to fetch historical prices

In [28]:
import yfinance as yf


@function_tool
def get_latest_stock_price(symbol: str) -> dict:
    """
    Fetch the current stock price for the given symbol.
    """

    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d") # Fetch the last 1 day of historical data
    latest_close = hist['Close'].iloc[-1]
    latest_date = hist.index[-1]

    return {'price': latest_close, 'date': latest_date}

In [ ]:
ticker = yf.Ticker("NVDA")
hist = ticker.history(period="1d")
hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-08-15 00:00:00-04:00,181.759995,181.899994,178.044693,178.919998,65930518,0.0,0.0


#### Agent

##### Define your agent

In [29]:
MODEL = "gpt-4.1"
stock_price_agent = Agent(
    name="Stock Price Agent",
    instructions="Fetch Historical prices for a given stock symbol",
    model = MODEL,
    tools = [get_latest_stock_price],
)

# Using the asynchronous runner inside Colab environment since an event loop is already running
output = await Runner.run(
    starting_agent=stock_price_agent,
    input="What is today's date, and what is the current stock price of Nvidia?",
)

print(output.final_output)

Today's date is November 25, 2026. The current stock price of Nvidia (NVDA) is $149.30.


##### Run your agent

In [30]:
output = await Runner.run(
    starting_agent=stock_price_agent,
    input="What is today's date, and what is the current stock price of Nvidia?",
)

print(output.final_output)

Today's date is November 25, 2026. The current stock price of Nvidia (NVDA) is $149.30.


### News Agent

In [41]:
from newsapi import NewsApiClient
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

from datetime import date, timedelta

#### Tool

Here I defined two tools:
* One fetching latest news from NewsAPI ==> **search_news**
* The other is summarizing these news, using an LLM ==> **summarize_news_api**

In [ ]:
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

Response(id='resp_03379e1e930c147a0068fce52110fc81969249b8397d011b96', created_at=1761404193.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_03379e1e930c147a0068fce52233088196a7abdb2672aa2e42', content=[ResponseOutputText(annotations=[], text='As of the **market close on June 14, 2024**, the last price of **NVIDIA Corporation (NVDA)** was **$130.98 USD** per share.\n\n*Please note*: Stock prices are subject to change during market hours. For the most current price, please check a reliable financial news source or your brokerage account.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, reasoning=Reasoning(effort

In [45]:
from datetime import date, timedelta
END_DATE = date.today()
START_DATE = date.today() - timedelta(days=7)

def _search_news_internal(ticker: str, num_articles: int = 3, to_datetime: date = END_DATE, from_datetime: date = START_DATE) -> str:
  """
  Internal function that gets the most recent news of a stock or an instrument

  Args:
  ticker (str): the stock ticker to be given to NEWSAPI
  num_articles(int): the number of news articles to collect
  to_datetime(date): Today's date
  from_datetime(date): Date from which to collect the news
  """

  all_articles = newsapi.get_everything(q=ticker,
                                          from_param=from_datetime,
                                          to=to_datetime,
                                          language='en',
                                          sort_by='relevancy',
                                          page_size=num_articles)

  news_concat = [
      f"{article['title']}, {article['description']}, {article['publishedAt']},{article['content'][0:100]}"
      for article in all_articles['articles']
  ]

  return (".\n").join(news_concat)


@function_tool
def search_news(ticker: str, num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Get the most recent news of a stock or an instrument

  Args:
  ticker (str): the stock ticker to be given to NEWSAPI
  num_articles (int): Number of news article to collect
  to_datetime (date): Today's date
  from_datetime (date): Date from which to collect the news (7 days before today for example)
  """
  return _search_news_internal(ticker, num_articles, to_datetime,from_datetime)

In [39]:
END_DATE = date.today()
START_DATE = date.today() - timedelta(days=7)
@function_tool
def summarize_news_api(ticker: str,num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Summarize the news of a given stock or an instrument

  Args:
    ticker (str): the stock ticker to be given to NEWSAPI
    num_articles (int): Number of news article to collect
    to_datetime (date): Today's date
    from_datetime (date): Date from which to collect the news (7 days before today for example)

  """

  news = _search_news_internal(ticker, num_articles, to_datetime,from_datetime)

  prompt = f"Summarize the following text by extracting the key insights: {news}"


  response = client.responses.create(
      model="gpt-4.1",
      input = prompt
  )

  return response.output[0].context[0].text

#### Running NewsAPI itself without LLM/Agent

In [46]:
num_articles = 3
to_datetime = END_DATE
from_datetime = START_DATE
ticker = " NVDA"

all_articles = newsapi.get_everything(q=ticker,
                                            from_param=from_datetime,
                                            to=to_datetime,
                                            language='en',
                                            sort_by='relevancy',
                                            page_size=num_articles)

news_concat =  [
    f"{article['title']}, {article['description']}, {article['publishedAt']},{article['content'][0:100]}"
    for article in all_articles['articles']
    ]

news = (".\n").join(news_concat)

prompt = f"Summarize the following text by extracting the key insights: {news}"
response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

print(response.output[0].content[0].text)

**Key Insights:**

1. **Jensen Huang Reinstated as Nvidia CEO:**  
   - Nvidia’s board announced Jensen Huang's reinstatement as CEO and president.
   - The decision followed a series of discussions with Huang.
   - Nvidia stock surged by more than 30% after the news.

2. **Nvidia Stock Activity by Officials:**  
   - There was significant trading activity in Nvidia stock around the period of Huang's resignation and reinstatement.
   - Former President Trump and 37 members of Congress engaged in 43 Nvidia stock trades during this time.
   - Transparency concerns exist regarding stock trading by Washington insiders around major corporate events.

3. **Broader AI Sector Volatility:**  
   - U.S. stocks continued to sell off despite Anthropic’s IPO.
   - The market remains uncertain about artificial intelligence safety and financing, contributing to stock volatility even as some major IPOs succeed.

These points capture the leadership change at Nvidia, the associated stock market impact, 

In [ ]:
news

'Nvidia Stock at 8% of the S&P 500 Index Is a Big Problem for Investors. Let’s Do the Math., One single stock essentially dictates our financial lives these days., 2025-10-21T14:42:22Z,Remember when energy stocks dominated the S&amp;P 500 Index ($SPX), and one company, Exxon Mobil (XO.\nCan Nvidia Stock Hit $300 in 2025?, Will Nvidia’s unstoppable AI momentum drive the stock past $300 by the end of 2025?, 2025-10-22T13:00:02Z,When it comes to artificial intelligence (AI), one name that towers above the rest is Nvidia (NVDA)..\nNVIDIA Corporation (NVDA) And Taiwan Semiconductor Manufacturing (TSM) Produces the First Blackwell Wafer in US, \u200bNVIDIA Corporation (NASDAQ:NVDA) is one of the Best Revenue Growth Stocks to Invest In. On October 17, NVIDIA Corporation (NASDAQ:NVDA) announced reaching a..., 2025-10-22T09:17:19Z,NVIDIA Corporation (NASDAQ:NVDA) is one of the Best Revenue Growth Stocks to Invest In. On October 1'

In [ ]:
#Another call to see if I have a consistent output
prompt = f"Summarize the following text by extracting the key insights: {news}"
response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

In [47]:
print(response.output[0].content[0].text)

**Key Insights:**

1. **Jensen Huang Reinstated as Nvidia CEO:**  
   - Nvidia’s board announced Jensen Huang's reinstatement as CEO and president.
   - The decision followed a series of discussions with Huang.
   - Nvidia stock surged by more than 30% after the news.

2. **Nvidia Stock Activity by Officials:**  
   - There was significant trading activity in Nvidia stock around the period of Huang's resignation and reinstatement.
   - Former President Trump and 37 members of Congress engaged in 43 Nvidia stock trades during this time.
   - Transparency concerns exist regarding stock trading by Washington insiders around major corporate events.

3. **Broader AI Sector Volatility:**  
   - U.S. stocks continued to sell off despite Anthropic’s IPO.
   - The market remains uncertain about artificial intelligence safety and financing, contributing to stock volatility even as some major IPOs succeed.

These points capture the leadership change at Nvidia, the associated stock market impact, 

#### Agent: Fetch News

In [ ]:
from agents import Agent, Runner, function_tool

In [55]:
MODEL = "gpt-4o-mini"
news_agent = Agent(
    name="News Agent",
    instructions="Get the latest news of a stock or an instrument using only the tools associated with the agent",
    model = MODEL,
    tools=[search_news],
)

In [54]:
output = await Runner.run(
    starting_agent=news_agent,
    input="Give me the latest news about Apple.",
)

print(output.final_output)

I couldn’t retrieve any Apple-related headlines: the AAPL search returned unrelated Nvidia stories, and a second search for “Apple” returned no results. I can’t reliably summarize Apple’s latest news from this feed.


In [49]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="give me the latest news about Nvidia?",
)

print(output.final_output)

RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.

In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Provide the latest news about Apple and Nvidia.",
)

print(output.final_output)

### Latest News on Apple (AAPL)

1. **Apple’s Stock Hits New All-Time High**  
   - **Date**: October 20, 2025  
   - Apple’s stock price recently reached an all-time high of approximately $263 per share, driven by surging iPhone 17 sales.

2. **Apple Stockholders Hit $850 Billion Jackpot**  
   - **Date**: October 22, 2025  
   - Over the past ten years, Apple stock has returned a substantial $847 billion to investors through dividends and buybacks.

3. **Apple Shines in Megacap Earnings Reports**  
   - **Date**: October 24, 2025  
   - Apple is part of a lineup of megacap hyperscalers making headlines in the latest earnings reports, holding near a fresh buy point.

### Latest News on Nvidia (NVDA)

1. **Nvidia Stock at 8% of the S&P 500 Index**  
   - **Date**: October 21, 2025  
   - The proportion of Nvidia stock in the S&P 500 Index poses challenges for investors, highlighting its significant market influence.

2. **Can Nvidia Stock Hit $300 in 2025?**  
   - **Date**: October 22

#### Agent: Summarize news

In [ ]:
MODEL = "gpt-4o-mini"
news_agent = Agent(
    name="News Agent",
    instructions="Get a summary about the latest news of a stock or an instrument using only the tools associated with the agent.",
    # instructions="Give me a summary about the latest news about OpenAI?",
    model = MODEL,
    tools=[summarize_news_api],
)

In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Summarize the latest news of Apple?",
)

print(output.final_output)

Here are the latest insights regarding Apple (AAPL):

- **Stock Performance**: Apple’s stock has reached an all-time high of around $263 per share, attributed to strong sales of the iPhone 17.
  
- **Investor Returns**: Over the last decade, Apple has returned about $847 billion to investors through dividends and stock buybacks.

- **Analyst Ratings**: Wells Fargo has raised its price target for Apple to $290, reaffirming an “Overweight” rating and marking it as one of the top investment options among Fortune 500 companies as of late 2025.


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Give me a summary of the latest news of NVIDIA?",
)

print(output.final_output)

Here's a summary of the latest news regarding NVIDIA:

1. **Market Dominance**: Nvidia now makes up about 8% of the S&P 500 Index, raising systemic risks similar to when Exxon Mobil dominated the index. This concentration means the index's performance is heavily influenced by Nvidia, which could lead to volatility.

2. **AI-Driven Growth Expectations**: As a leader in the AI sector, there's speculation that Nvidia’s stock might exceed $300 by the end of 2025. The company's strong momentum in AI is crucial for its revenue and stock performance.

3. **Technological Advancements**: Nvidia, in collaboration with Taiwan Semiconductor Manufacturing Company (TSMC), has created the first Blackwell wafer in the US, highlighting its innovation and commitment to US manufacturing.

4. **Revenue Growth Leadership**: Nvidia continues to be recognized for exceptional revenue growth, with strong results and technological advancements reported as of October 2025.

Overall, while Nvidia's concentration 